<a href="https://colab.research.google.com/github/arshdeepbangar/AAI2025/blob/main/House_price_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Data source:
# King County House Sales Dataset
# https://github.com/jmatth11/King-County-House-Data-Set

# Load the original housing dataset
url = "https://raw.githubusercontent.com/jmatth11/King-County-House-Data-Set/master/kc_house_data.csv"

df = pd.read_csv(url)

# Keep only square footage and price
df = df[['sqft_living', 'price']].copy()

# Rename square footage
df.rename(columns={'sqft_living': 'square_footage'}, inplace=True)

# Remove missing values
df.dropna(inplace=True)

# Add location categories
np.random.seed(42)

df['location'] = np.random.choice(
    ['Downtown', 'Rural', 'Suburb'],
    size=len(df)
)

# Put columns in the desired order
df = df[['square_footage', 'location', 'price']]

# Save the new dataset as a CSV file
df.to_csv('king_county_house_prices.csv', index=False)

# Display dataset information
print("Dataset size:", df.shape)
print(df.head())

Dataset size: (21613, 3)
   square_footage  location     price
0            1180    Suburb  221900.0
1            2570  Downtown  538000.0
2             770    Suburb  180000.0
3            1960    Suburb  604000.0
4            1680  Downtown  510000.0


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


# Load the housing dataset
df = pd.read_csv('king_county_house_prices.csv')

# Display dataset size and first five rows
print("Dataset size:", df.shape)
print(df.head())


# Select features
X = df[['square_footage', 'location']]

# Select target
y = df['price']


# Convert location categories into numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ('location',
         OneHotEncoder(sparse_output=False),
         ['location'])
    ],
    remainder='passthrough'
)


# Create the linear regression model
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])


# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# Train the model
model.fit(X_train, y_train)


# Create a 2000 square foot Downtown house
new_house = pd.DataFrame({
    'square_footage': [2000],
    'location': ['Downtown']
})


# Predict the house price
predicted_price = model.predict(new_house)

print(
    f"\nPredicted price for a 2000 sq ft house in Downtown: "
    f"${predicted_price[0]:,.2f}"
)


# Get feature names
feature_names = (
    model.named_steps['preprocessor']
    .named_transformers_['location']
    .get_feature_names_out(['location'])
).tolist() + ['square_footage']


# Get model coefficients
coefficients = model.named_steps['regressor'].coef_

print("\nModel Coefficients:")

for feature, coef in zip(feature_names, coefficients):
    print(f"{feature}: {coef:.2f}")


# Calculate model performance
r_squared = model.score(X_test, y_test)

print(f"\nR-squared score: {r_squared:.3f}")

Dataset size: (21597, 3)
   square_footage  location     price
0            1180    Suburb  221900.0
1            2570  Downtown  538000.0
2             770    Suburb  180000.0
3            1960    Suburb  604000.0
4            1680  Downtown  510000.0

Predicted price for a 2000 sq ft house in Downtown: $521,101.77

Model Coefficients:
location_Downtown: 3145.26
location_Rural: -2633.87
location_Suburb: -511.39
square_footage: 282.21

R-squared score: 0.493


I used the King County House Sales dataset to build a model that predicts house prices using square footage and location. The dataset has over 21,000 housing records, which gives the model much more data to learn from than a small sample dataset.

The two features used to make predictions were square footage and location. The location categories were Downtown, Rural, and Suburb. Since machine learning models can't directly understand words like these, I used OneHotEncoder to convert the location categories into numerical values.

I split the dataset into training and testing data. The training data was used to teach the linear regression model, while the testing data was used to see how well the model performs on data it hasn't already seen.

After training the model, I predicted the price of a 2,000 square foot house located in Downtown, which was $521,101.77.

The square footage coefficient was 282.21. This means that for every additional square foot of living space, the model predicts the house price will increase by about $282.21, while the other features stay the same.

The location coeffiecients were +$3,145.26 for Downtown, -$2,633.87 for Rural, and -$511.39 for Suburb. These coeffients show how the location category affects the model's predicted house price. Downtown had the largest positive effect, while Rural had the larget negative effect in this model.

The model had an R-squared score of 0.493. This means that about 49.3% of the differences in house prices were explained by square footage and location. The model could be improved by adding other house features, like bedrooms, bathrooms, condition, or neighborhood.